In [1]:
import sys
sys.path.append("..")

import pandas as pd

from src.data import load_dataset
from src.embedder import record_to_text, get_or_compute_embeddings
from src.search import search_all_questions
from src.metrics import evaluate
from src.config import TOP_K
from sentence_transformers import SentenceTransformer

corpus, questions, categories = load_dataset()
print(f"Корпус: {len(corpus)} фрагментов")
print(f"Вопросы: {len(questions)} вопросов")

corpus_texts = [record_to_text(item) for item in corpus]

Корпус: 200 фрагментов
Вопросы: 25 вопросов


In [2]:
MODEL_NAMES = {
    "MiniLM":   "paraphrase-multilingual-MiniLM-L12-v2",
    "mpnet":    "paraphrase-multilingual-mpnet-base-v2",
    "e5-small": "intfloat/multilingual-e5-small",
}

models = {}
embeddings = {}

for short_name, model_name in MODEL_NAMES.items():
    print(f"\nЗагружаю {short_name}...")
    models[short_name] = SentenceTransformer(model_name)
    embeddings[short_name] = get_or_compute_embeddings(corpus_texts, model_name)

print("\nВсе эмбеддинги готовы")


Загружаю MiniLM...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

loading embedding from the cache...

Загружаю mpnet...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

loading embedding from the cache...

Загружаю e5-small...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

loading embedding from the cache...

Все эмбеддинги готовы


In [3]:
ru_questions = [q for q in questions if q["language"] == "ru"]
en_questions = [q for q in questions if q["language"] == "en"]

print(f"Русских вопросов: {len(ru_questions)}")
print(f"Английских вопросов: {len(en_questions)}")

rows = []
for short_name, model_name in MODEL_NAMES.items():
    model = models[short_name]
    corpus_emb = embeddings[short_name]

    for lang_name, lang_questions in [("Русский", ru_questions), ("English", en_questions)]:
        search_outputs = search_all_questions(
            questions=lang_questions,
            model=model,
            corpus=corpus,
            corpus_embeddings=corpus_emb,
            top_k=TOP_K,
        )
        metrics = evaluate(search_outputs, k=TOP_K)

        rows.append({
            "Модель": short_name,
            "Язык": lang_name,
            "Precision@3": round(metrics["precision_at_3"], 4),
            "MRR": round(metrics["mrr"], 4),
            "Вопросов": metrics["num_questions"],
        })

df = pd.DataFrame(rows)
display(df)

Русских вопросов: 15
Английских вопросов: 10


,Модель,Язык,Precision@3,MRR,Вопросов
0,MiniLM,Русский,0.8000,0.5444,15
1,MiniLM,English,0.9000,0.5833,10
2,mpnet,Русский,0.9333,0.5889,15
3,mpnet,English,0.9000,0.7000,10
4,e5-small,Русский,0.9333,0.7889,15
5,e5-small,English,1.0000,0.7500,10


In [4]:
best_row = df[df["Язык"] == "Русский"].sort_values("MRR", ascending=False).iloc[0]
worst_row = df[df["Язык"] == "Русский"].sort_values("MRR", ascending=True).iloc[0]

print("=" * 70)
print("Вывод:")
print("=" * 70)
print(f"Лучшая модель на русском: {best_row['Модель']}")
print(f"  Precision@3 = {best_row['Precision@3']}, MRR = {best_row['MRR']}")
print(f"Худшая модель на русском: {worst_row['Модель']}")
print(f"  Precision@3 = {worst_row['Precision@3']}, MRR = {worst_row['MRR']}")

gap = round(best_row["MRR"] - worst_row["MRR"], 4)
print(f"\nРазрыв по MRR между лучшей и худшей: {gap}")
print(f"\nЕсли вопросы только на русском — ожидаемый Precision@3 лучшей модели: {best_row['Precision@3']}")
print("=" * 70)

Вывод:
Лучшая модель на русском: e5-small
  Precision@3 = 0.9333, MRR = 0.7889
Худшая модель на русском: MiniLM
  Precision@3 = 0.8, MRR = 0.5444

Разрыв по MRR между лучшей и худшей: 0.2445

Если вопросы только на русском — ожидаемый Precision@3 лучшей модели: 0.9333
